In [5]:
#pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
#!pip install pillow pandas

In [7]:

import os
import time
import torch
import torchvision
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load models (pretrained COCO)
model_frcnn = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")
model_retinanet = torchvision.models.detection.retinanet_resnet50_fpn(weights="DEFAULT")

model_frcnn.to(device).eval()
model_retinanet.to(device).eval()

# Image transform
transform = transforms.Compose([
    transforms.ToTensor()
])

# Folder with your images
IMAGE_DIR = "images"   # put your 10+ images here
image_files = [f for f in os.listdir(IMAGE_DIR) if f.lower().endswith((".jpg", ".jpeg", ".png"))]

print(f"Found {len(image_files)} images.")



Found 10 images.


In [8]:
def compute_average_brightness(pil_image):
    # Convert to grayscale and take mean pixel value (0–255)
    gray = pil_image.convert("L")
    arr = np.array(gray, dtype=np.float32)
    return arr.mean()


In [9]:
results = []

score_threshold = 0.5  # you can adjust this

for img_name in image_files:
    img_path = os.path.join(IMAGE_DIR, img_name)
    pil_img = Image.open(img_path).convert("RGB")

    # non-DL feature: average brightness
    avg_brightness = compute_average_brightness(pil_img)

    img_tensor = transform(pil_img).to(device)  # [C, H, W]

    # -------------- Faster R-CNN --------------
    with torch.no_grad():
        start_time = time.time()
        outputs_frcnn = model_frcnn([img_tensor])[0]  # list -> first element
        elapsed_frcnn = time.time() - start_time

    scores_frcnn = outputs_frcnn["scores"].detach().cpu().numpy()
    labels_frcnn = outputs_frcnn["labels"].detach().cpu().numpy()

    keep_frcnn = scores_frcnn >= score_threshold
    num_objs_frcnn = keep_frcnn.sum()
    avg_score_frcnn = scores_frcnn[keep_frcnn].mean() if num_objs_frcnn > 0 else 0.0

    # -------------- RetinaNet --------------
    with torch.no_grad():
        start_time = time.time()
        outputs_retina = model_retinanet([img_tensor])[0]
        elapsed_retina = time.time() - start_time

    scores_retina = outputs_retina["scores"].detach().cpu().numpy()
    labels_retina = outputs_retina["labels"].detach().cpu().numpy()

    keep_retina = scores_retina >= score_threshold
    num_objs_retina = keep_retina.sum()
    avg_score_retina = scores_retina[keep_retina].mean() if num_objs_retina > 0 else 0.0

    # Store row-wise results for each model
    results.append({
        "image": img_name,
        "brightness": round(avg_brightness, 2),
        "model": "Faster R-CNN",
        "num_objects": int(num_objs_frcnn),
        "avg_score": round(float(avg_score_frcnn), 3),
        "time_sec": round(float(elapsed_frcnn), 3)
    })

    results.append({
        "image": img_name,
        "brightness": round(avg_brightness, 2),
        "model": "RetinaNet",
        "num_objects": int(num_objs_retina),
        "avg_score": round(float(avg_score_retina), 3),
        "time_sec": round(float(elapsed_retina), 3)
    })

df_results = pd.DataFrame(results)
print(df_results)


          image  brightness         model  num_objects  avg_score  time_sec
0    img 7.jpeg  109.389999  Faster R-CNN            4      0.750     0.816
1    img 7.jpeg  109.389999     RetinaNet            1      0.825     0.632
2    img 6.jpeg  109.980003  Faster R-CNN            1      0.645     0.551
3    img 6.jpeg  109.980003     RetinaNet            1      0.500     0.524
4     img 5.jpg   93.050003  Faster R-CNN            1      0.979     0.830
5     img 5.jpg   93.050003     RetinaNet            1      0.906     0.676
6    img 1.jpeg  109.940002  Faster R-CNN            2      0.835     0.554
7    img 1.jpeg  109.940002     RetinaNet            2      0.745     0.531
8    img 3.jpeg   92.010002  Faster R-CNN            3      0.768     0.458
9    img 3.jpeg   92.010002     RetinaNet            1      0.506     0.417
10  img 10.jpeg  150.250000  Faster R-CNN            1      0.966     0.457
11  img 10.jpeg  150.250000     RetinaNet            1      0.874     0.426
12   img 2.j